<a href="https://colab.research.google.com/github/MoisesNb7/conociendo_monedas_del_mundo/blob/main/conocimiento_monedas_del_mundo_MNB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🪙 **Reto: Conociendo las Monedas del Mundo**

## 📝 Introducción y Objetivos
Este cuaderno consolida y valida un catálogo internacional de monedas comparando información extraída de tres fuentes distintas:
1. **API REST pública:** Coinbase (`/v2/currencies`).
2. **Web Scraping:** Extracción con `BeautifulSoup` de la tabla principal de Wikipedia (*List of circulating currencies*).
3. **Documentos no estructurados:** Lectura y parsing con `tabula` del archivo `CurrencyCodes2.pdf`.

**Objetivo:** Extraer, normalizar y cruzar los datos de las tres fuentes para obtener la lista final de monedas comunes validadas y ordenadas por su código de tres caracteres.

---

# **1. Consulta la información de las monedas del mundo que nos provee la API: https://api.coinbase.com/v2/currencies**

In [113]:
!pip install requests
!pip install beautifulsoup4
!pip install pypdf

In [114]:
import requests

url = "https://api.coinbase.com/v2/currencies"

response = requests.get(url)

datos = response.json()

lista_monedas_json = []

for i in datos["data"]:
  # 1. Construye una lista de tuplas de dos elementos, donde el primer elemento
  # de la tupla es el código de la moneda y el segundo es el nombre
  tupla = (i['id'].lower(), i['name'].lower())
  lista_monedas_json.append(tupla)

# 2. Manda a imprimir en pantalla la lista creada.
print("Lista de elementos:")
print(lista_monedas_json)

# 3. Manda a imprimir el tamaño de la lista.
print(f"Tamaño de la lista: {len(lista_monedas_json)}")



Lista de elementos:
[('aed', 'united arab emirates dirham'), ('afn', 'afghan afghani'), ('all', 'albanian lek'), ('amd', 'armenian dram'), ('ang', 'netherlands antillean gulden'), ('aoa', 'angolan kwanza'), ('ars', 'argentine peso'), ('aud', 'australian dollar'), ('awg', 'aruban florin'), ('azn', 'azerbaijani manat'), ('bam', 'bosnia and herzegovina convertible mark'), ('bbd', 'barbadian dollar'), ('bdt', 'bangladeshi taka'), ('bgn', 'bulgarian lev'), ('bhd', 'bahraini dinar'), ('bif', 'burundian franc'), ('bmd', 'bermudian dollar'), ('bnd', 'brunei dollar'), ('bob', 'bolivian boliviano'), ('brl', 'real'), ('bsd', 'bahamian dollar'), ('btn', 'bhutanese ngultrum'), ('bwp', 'botswana pula'), ('byn', 'belarusian ruble'), ('byr', 'belarusian ruble'), ('bzd', 'belize dollar'), ('cad', 'canadian dollar'), ('cdf', 'congolese franc'), ('chf', 'swiss franc'), ('clf', 'unidad de fomento'), ('clp', 'chilean peso'), ('cny', 'chinese renminbi yuan'), ('cop', 'colombian peso'), ('crc', 'costa rican 

# **2. Consulta la información relativa a las monedas del mundo del sitio https://en.wikipedia.org/wiki/List_of_circulating_currencies**

Considera que en el sitio hay más de una tabla, por lo que solo necesitas la información de la primera.

Algunos de los códigos de moneda (de 3 caracteres) tienen referencias, es decir, después de los 3 caracteres tienen un número o una letra entre corchetes (por ejemplo: CKD[F]). Quita la información entre corchetes, dejando solo los 3 caracteres del código.

No todos los renglones (<tr>) tienen el mismo número de elementos (<td>). Ten cuidado en cómo extraer los elementos requeridos.

Construye una lista de tuplas de dos elementos, donde el primer elemento de la tupla sea el código de la moneda y el segundo sea el nombre.

In [115]:
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_circulating_currencies"

# Agente
headers = {
    "User-Agent": "MiProyectoEscolar/1.0 (contacto@ejemplo.com) Python-requests"
}

page = requests.get(url, headers=headers)

soup = BeautifulSoup(page.content, "html.parser")

# Seleccionamos la primera tabla
tabla = soup.find("table", class_="wikitable")

print(tabla)

<table class="wikitable sortable sticky-header mw-collapsible" id="mwWQ">
<tbody id="mwWg"><tr id="mwWw">
<th id="mwXA" scope="col"><a href="https://en.wikipedia.org/wiki/List_of_sovereign_states" id="mwXQ" rel="mw:WikiLink" title="List of sovereign states">State</a> or <a href="https://en.wikipedia.org/wiki/Dependent_territory" id="mwXg" rel="mw:WikiLink" title="Dependent territory">territory</a><sup about="#mwt27" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"CIA"},"body":{"id":"mw-reference-text-cite_note-CIA-5"}}' id="cite_ref-CIA_5-0" rel="dc:references" typeof="mw:Extension/ref"><a href="#cite_note-CIA-5" id="mwXw"><span class="mw-reflink-text" id="mwYA"><span class="cite-bracket" id="mwYQ">[</span>2<span class="cite-bracket" id="mwYg">]</span></span></a></sup></th>
<th id="mwYw" scope="col"><a href="https://en.wikipedia.org/wiki/Currency" id="mwZA" rel="mw:WikiLink" title="Currency">Currency</a><sup about="#mwt28" class="mw-ref reference" data-mw='{"name":"ref

In [116]:
import re
import pandas as pd

lista_monedas_wikipedia = []

# Obtener solo la primera tabla
tabla = soup.find("table", {"class": "wikitable"})

for fila in tabla.find_all("tr"):
    celdas = fila.find_all("td")

    # Validar celdas
    if len(celdas) >= 4:
        nombre_texto = celdas[1].text.strip()
        codigo_moneda = celdas[3].text.strip()

        # Eliminar referencias en corchetes
        codigo_limpio = re.sub(r"\[.*?\]", "", codigo_moneda).strip()

        # Filtrar códigos de 3 letras
        if len(codigo_limpio) == 3:
            tupla = (codigo_limpio.lower(), nombre_texto.lower())
            lista_monedas_wikipedia.append(tupla)

# Exportar datos a Excel
#dataframe = pd.DataFrame(lista_monedas_wikipedia, columns=["Código", "Nombre"])
#dataframe.to_excel("lista_monedas_wikipedia.xlsx", index=False)

print("Tamaño de la lista:", len(lista_monedas_wikipedia))
print(lista_monedas_wikipedia)

#dataframe

Tamaño de la lista: 226
[('afn', 'afghan afghani'), ('eur', 'euro'), ('all', 'albanian lek'), ('dzd', 'algerian dinar'), ('eur', 'euro'), ('aoa', 'angolan kwanza'), ('xcd', 'eastern caribbean dollar'), ('xcd', 'eastern caribbean dollar'), ('ars', 'argentine peso'), ('amd', 'armenian dram'), ('awg', 'aruban florin'), ('shp', 'saint helena pound'), ('aud', 'australian dollar'), ('eur', 'euro'), ('azn', 'azerbaijani manat'), ('bsd', 'bahamian dollar'), ('bhd', 'bahraini dinar'), ('bdt', 'bangladeshi taka'), ('bbd', 'barbadian dollar'), ('byn', 'belarusian ruble'), ('eur', 'euro'), ('bzd', 'belize dollar'), ('xof', 'west african cfa franc'), ('bmd', 'bermudian dollar'), ('btn', 'bhutanese ngultrum[f]'), ('bob', 'bolivian boliviano'), ('usd', 'united states dollar[g]'), ('bam', 'bosnia and herzegovina convertible mark'), ('bwp', 'botswana pula'), ('brl', 'brazilian real'), ('usd', 'united states dollar'), ('bnd', 'brunei dollar'), ('eur', 'euro'), ('xof', 'west african cfa franc'), ('bif', 

# **3. Extrae la información de las monedas del mundo del documento CurrencyCodes2.pdf y crea, de igual manera, una lista de tuplas de dos elementos, donde el primer elemento de la tupla sea el código de la moneda y el segundo sea el nombre**

In [117]:
!pip install tabula-py[jpype]
import tabula

In [118]:
# Lee todas las páginas del PDF
tabla = tabula.read_pdf("CurrencyCodes2.pdf", pages="all", pandas_options={'header': None})

lista_monedas_pdf = []

# Recorre cada página y cada fila para extraer los datos
for pagina in tabla:
    for i in range(len(pagina)):
        # Omite la primera pag
        if pagina is tabla[0] and i == 0:
            continue

        nombre = pagina.iloc[i, 0]
        codigo = str(pagina.iloc[i, 1])[:3]

        lista_monedas_pdf.append((codigo.lower(), nombre.lower()))

# Muestra el resultado final y el total de registros
print(lista_monedas_pdf)
print("Total de elementos:", len(lista_monedas_pdf))

[('afn', 'currency of afganistan is the afghani'), ('all', 'currency of albania is the lek'), ('dzd', 'currency of alergia is the algerian dinar'), ('usd', 'currency of american samoa is the us dollar'), ('eur', 'currency of andorra is the euro'), ('aoa', 'currency of angola is the kwanza'), ('xcd', 'currency of anguilla is the east caribbean dollar'), ('xcd', 'currency of antigua and barbuda is the east caribbean dollar'), ('ars', 'currency of argentina is the argentine peso'), ('amd', 'currency of armenia is the armenian dram'), ('awg', 'currency of aruba is the aruban florin'), ('aud', 'currency of australia is the australian dollar'), ('eur', 'currency of austria is the euro'), ('azn', 'currency of azerbaijan is the manat'), ('bsd', 'currency of bahamas is the bahamian dollar'), ('bhd', 'currency of bahrain is the bahraini dinar'), ('bdt', 'currency of bangladesh is the taka'), ('bbd', 'currency of barbados is the barbados dollar'), ('byn', 'currency of belarus is the belarussian r

# **4. De las tres listas obtenidas, construye una lista nueva con los elementos que existan en las tres listas obtenidas (es decir, los elementos comunes).**

In [123]:
# Extraemos los códigos de B y C
codigos_b = {t[0] for t in lista_monedas_wikipedia}
codigos_c = {t[0] for t in lista_monedas_pdf}

# Filtramos la primera lista por esos códigos
lista_comunes = [t for t in lista_monedas_json if t[0] in codigos_b and t[0] in codigos_c]

print(lista_comunes)
print("Total de elementos comunes:", len(lista_comunes))

[('aed', 'united arab emirates dirham'), ('afn', 'afghan afghani'), ('all', 'albanian lek'), ('amd', 'armenian dram'), ('aoa', 'angolan kwanza'), ('ars', 'argentine peso'), ('aud', 'australian dollar'), ('awg', 'aruban florin'), ('azn', 'azerbaijani manat'), ('bam', 'bosnia and herzegovina convertible mark'), ('bbd', 'barbadian dollar'), ('bdt', 'bangladeshi taka'), ('bhd', 'bahraini dinar'), ('bif', 'burundian franc'), ('bmd', 'bermudian dollar'), ('bnd', 'brunei dollar'), ('bob', 'bolivian boliviano'), ('brl', 'real'), ('bsd', 'bahamian dollar'), ('btn', 'bhutanese ngultrum'), ('bwp', 'botswana pula'), ('byn', 'belarusian ruble'), ('bzd', 'belize dollar'), ('cad', 'canadian dollar'), ('cdf', 'congolese franc'), ('chf', 'swiss franc'), ('clp', 'chilean peso'), ('cny', 'chinese renminbi yuan'), ('cop', 'colombian peso'), ('crc', 'costa rican colón'), ('cup', 'cuban peso'), ('cve', 'cape verdean escudo'), ('czk', 'czech koruna'), ('djf', 'djiboutian franc'), ('dkk', 'danish krone'), ('d

Finalmente, imprime la lista que contiene los elementos comunes ordenada por el código de moneda, seguido de la impresión del número de elementos en esta última lista de elementos comunes.

In [124]:
cont = 0
lista_comunes.sort()


for i in lista_comunes:
    cont = cont + 1
    print(cont, i)
print("Total de elementos comunes:",len(lista_comunes))


1 ('aed', 'united arab emirates dirham')
2 ('afn', 'afghan afghani')
3 ('all', 'albanian lek')
4 ('amd', 'armenian dram')
5 ('aoa', 'angolan kwanza')
6 ('ars', 'argentine peso')
7 ('aud', 'australian dollar')
8 ('awg', 'aruban florin')
9 ('azn', 'azerbaijani manat')
10 ('bam', 'bosnia and herzegovina convertible mark')
11 ('bbd', 'barbadian dollar')
12 ('bdt', 'bangladeshi taka')
13 ('bhd', 'bahraini dinar')
14 ('bif', 'burundian franc')
15 ('bmd', 'bermudian dollar')
16 ('bnd', 'brunei dollar')
17 ('bob', 'bolivian boliviano')
18 ('brl', 'real')
19 ('bsd', 'bahamian dollar')
20 ('btn', 'bhutanese ngultrum')
21 ('bwp', 'botswana pula')
22 ('byn', 'belarusian ruble')
23 ('bzd', 'belize dollar')
24 ('cad', 'canadian dollar')
25 ('cdf', 'congolese franc')
26 ('chf', 'swiss franc')
27 ('clp', 'chilean peso')
28 ('cny', 'chinese renminbi yuan')
29 ('cop', 'colombian peso')
30 ('crc', 'costa rican colón')
31 ('cup', 'cuban peso')
32 ('cve', 'cape verdean escudo')
33 ('czk', 'czech koruna')
3